# Multimodal LoRA pretraining experiment record

This is the record of the **multimodal LoRA experiments in which text and image were trained in the same run**. It is not the later text-only/image-only EMA-JEPA pilot notebook.

The scientific question was: can one shared LoRA pathway acquire useful common structure from unpaired text and images, while modality-private LoRA pathways retain information needed to generate each modality?

## Common architecture

The backbone has 8 Transformer blocks, width 384, learned absolute 1-D positions, and **no modality embedding**. Image inputs are a 16×24 grid of discrete VQ codes; text inputs are caption tokens.

In dense mode a selected linear layer has one ordinary trainable matrix. In strict no-base Tri-LoRA, there is no frozen dense weight in selected modules (`qkv`, `out_proj`, `mlp.0`, `mlp.3`). For a text token:

\[ y_T=\Delta_{shared}(x)+\Delta_{text}(x)+b_{shared}. \]

For an image token:

\[ y_I=\Delta_{shared}(x)+\Delta_{image}(x)+b_{shared}. \]

The shared and private adapters are low-rank affine paths. `shared_bias` is trainable; it is not a retained frozen base matrix.

## What “multimodal unpaired training” means here

At every epoch, text and image index orders are independently reshuffled. The data loader forms a carrier batch containing a text sub-batch and an image sub-batch. They contribute to the same optimizer update, but they are **not treated as matched pairs**: there is no text-image joint attention, correspondence target, contrastive pair loss, or image-caption reconstruction target.

For each modality separately, random eligible tokens are replaced by `[MASK]`; the model predicts their original discrete IDs using masked-token cross entropy. The shared LoRA parameters receive gradients from both modalities; text-private and image-private LoRA only receive their own modality's gradients.

\[L_{task}=\tfrac12(L_{diffusion,text}+L_{diffusion,image}).\]

## Core 70-epoch multimodal LoRA runs

| Name used in reports | Configuration | Purpose | Checkpoint |
|---|---|---|---|
| `plain_lora_no_stage` | shared + private from epoch 0 | Main unpaired LoRA reference | `outputs/pretraining_unpaired_lora_absolute_no_stage0_70e/best.pt` |
| `lora_stage0` | epochs 0–9: shared only; epoch 10+: shared + private | Test whether shared-first learning helps | `outputs/pretraining_unpaired_lora_absolute_stage0_70e/best.pt` |
| `lora_dann` | shared + private; DANN from epoch 0 | Encourage modality-invariant shared states | `outputs/pretraining_unpaired_lora_absolute_dann_70e/best.pt` |
| `lora_stage0_then_dann` | shared only for 10 epochs; then private + DANN | Combine shared-first training and DANN | `outputs/pretraining_unpaired_lora_absolute_stage0_then_dann_70e/best.pt` |
| `dense_unpaired` | one dense pathway, no shared/private split | Capacity/control baseline | `outputs/pretraining_unpaired_dense_absolute_70e/best.pt` |
| `dense_paired` | dense model with true image-caption pairs | Upper reference when correspondence is supplied | `outputs/pretraining_paired_dense_absolute_70e/best.pt` |

All six use the same tokenizer, dataset, position scheme, 70-epoch schedule, and no modality embeddings. The dense runs do not have a shared/private decomposition, so branch-specific probes do not apply to them.

## Stage 0

Stage 0 does **not** mean text and image become pairs. It only freezes private LoRA branches for the first 10 epochs:

\[\Delta_{text}=\Delta_{image}=0,\qquad y=\Delta_{shared}(x)+b_{shared}.\]

Both modalities therefore update the same shared LoRA adapters during this period. After epoch 10, private branches are enabled and normal separate modality losses continue.

## DANN experiments

DANN attaches a small discriminator to the pooled shared readout. Before the discriminator, each sample representation is L2-normalized:

\[\tilde s= s/(\lVert s\rVert_2+\epsilon).\]

The discriminator predicts text versus image. A gradient-reversal layer makes the shared LoRA update *confuse* this classifier:

\[L=L_{task}+0.1L_{DANN};\]

the discriminator minimizes its classification loss, while the shared pathway receives the reversed gradient. In `lora_dann` this begins at epoch 0; in `lora_stage0_then_dann` it begins at epoch 10 together with the private branches. DANN is an invariance pressure, not a paired alignment loss.

## SIGReg and the earlier multimodal JEPA runs

These were also multimodal: both text and image were processed independently in the same pretraining run and shared the shared-LoRA parameters and JEPA predictors.

| Name | JEPA target and layers | SIGReg | Checkpoint |
|---|---|---|---|
| `jepa_per_layer` | online clean stop-gradient target; per-layer shared token readouts | per layer | `outputs/pretraining_unpaired_lora_shared_jepa_sigreg_per_layer_absolute_70e/best.pt` |
| `jepa_normalized` | normalized-MSE, layers 2–4 | per layer | `outputs/pretraining_unpaired_lora_middle_l2_l4_normalized_jepa_sigreg_absolute_70e/best.pt` |
| `jepa_strong_normalized` | stronger balanced normalized-MSE, layers 2–4 | per layer | `outputs/pretraining_unpaired_lora_middle_l2_l4_strong_balanced_normalized_jepa_sigreg_absolute_70e/best.pt` |
| `jepa_strong_raw_l2` | stronger raw squared-L2, layers 2–4 | per layer | `outputs/pretraining_unpaired_lora_middle_l2_l4_strong_balanced_raw_l2_jepa_sigreg_absolute_70e/best.pt` |

For a masked position \(i\), their JEPA term is approximately:

\[ L_{JEPA}=\frac1{|M|}\sum_{i\in M} d(P_l(S_l(x_{masked})_i),\operatorname{sg}(S_l(x_{clean})_i)). \]

The earlier target was the same online model under `no_grad`, not an EMA teacher. SIGReg was an additional distributional anti-collapse regularizer; it does not by itself demonstrate semantic learning.

## Later single-modality EMA-JEPA pilots: separate experiment

The newer text-only and image-only EMA-JEPA pilots are **not** part of the multimodal runs above. They were intentionally separated to answer a narrower question: can JEPA produce robust representations in one modality before trying to align modalities?

They use a full EMA copy of the student as a clean target. The student is still the final diffusion model; the teacher is only a stop-gradient target. The image pilots now running mirror the text pilots: fixed JEPA weight 0.5 and dynamic gradient ratios 0.10, 0.25, 0.50.

In [ ]:
# Convenience: inspect the exact saved configuration of any core run.
from pathlib import Path
import yaml

ROOT = Path('/home/zd25e122/clevr_discrete_diffusion')
config = ROOT / 'configs/pretraining_unpaired_lora_absolute_stage0_then_dann_70e.yaml'
with config.open() as f:
    cfg = yaml.safe_load(f)
cfg
